# PrimeVul UniXcoder baseline (Google Colab)

This notebook fine-tunes `microsoft/unixcoder-base` on the **original PrimeVul release** and evaluates the official binary-classification baseline, including **P-C / P-V / P-B / P-R** on `primevul_test_paired.jsonl`.

It follows the published baseline as closely as practical:

- 512-token inputs
- 10 epochs
- learning rate `2e-5`
- 1,000 warmup optimizer steps
- effective training batch size 64
- one output logit trained with binary cross-entropy (BCE)
- threshold 0.5 for ordinary classification and paired metrics
- seed `123456`

The official paper reports for UniXcoder trained and tested on PrimeVul: **P-C 1.60, P-V 12.06, P-B 85.11, P-R 1.24** (percent). Exact replication is not guaranteed because CUDA, PyTorch, Transformers, batching, and checkpoint-selection versions can change results.

Before running: in Colab choose **Runtime → Change runtime type → A100 GPU**. Run cells from top to bottom. Public Hugging Face models do not require a token.

In [ ]:
# Install the small set of libraries used below. This should take a few minutes.
%pip install -q "transformers>=4.46,<5" "datasets>=3,<5" "accelerate>=1,<2" "gdown>=5,<6" "scikit-learn>=1.4,<2" "pandas>=2.1,<3" tensorboard

## Configuration and persistent storage

The dataset and all checkpoints are kept under `MyDrive/primevul_unixcoder_baseline`, so a Colab disconnect does not erase them. The model itself is downloaded automatically from Hugging Face.

`TRAIN_BATCH_SIZE=16` with four accumulation steps gives the published effective batch size of 64 while fitting a typical 40 GB Colab A100. Full FP32 is the default because the official baseline command did not enable mixed precision; `USE_BF16=True` is available if speed matters more than strict matching. Do not enable class weighting for the standard Table V baseline.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ----- Baseline configuration -----
MODEL_NAME = "microsoft/unixcoder-base"
SEED = 123456
MAX_LENGTH = 512
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WARMUP_STEPS = 1000
TRAIN_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch = 16 * 4 = 64
EVAL_BATCH_SIZE = 64
EVAL_AND_SAVE_STEPS = 1000
USE_BF16 = True  # much faster on A100; keep False only for closer FP32 matching
THRESHOLD = 0.5
RESUME_FROM_LAST_CHECKPOINT = True

PROJECT_DIR = Path("/content/drive/MyDrive/primevul_unixcoder_baseline")
DATA_DIR = PROJECT_DIR / "data"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
FINAL_MODEL_DIR = PROJECT_DIR / "best_model"
RESULTS_DIR = PROJECT_DIR / "results"
LOCAL_HF_CACHE = Path("/content/primevul_hf_cache")

for path in (DATA_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, RESULTS_DIR, LOCAL_HF_CACHE):
    path.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "No GPU detected. Select an A100 GPU in Colab Runtime settings."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Persistent project directory:", PROJECT_DIR)
print("Effective training batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

In [ ]:
# Download the six files from the original PrimeVul release once.
# Source: https://github.com/DLVulDet/PrimeVul#-primevul-dataset
import gdown

PRIMEVUL_FOLDER_URL = "https://drive.google.com/drive/folders/19iLaNDS0z99N8kB_jBRTmDLehwZBolMY"
REQUIRED_FILES = {
    "train": "primevul_train.jsonl",
    "validation": "primevul_valid.jsonl",
    "test": "primevul_test.jsonl",
    "test_paired": "primevul_test_paired.jsonl",
}


def locate_file(filename):
    matches = list(DATA_DIR.rglob(filename))
    return matches[0] if matches else None


missing = [name for name in REQUIRED_FILES.values() if locate_file(name) is None]
if missing:
    print("Downloading the original PrimeVul release. This is roughly 450 MB...")
    gdown.download_folder(
        url=PRIMEVUL_FOLDER_URL,
        output=str(DATA_DIR),
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
    )
else:
    print("Dataset already exists in Google Drive; skipping download.")

DATA_FILES = {split: locate_file(name) for split, name in REQUIRED_FILES.items()}
missing = [name for split, name in REQUIRED_FILES.items() if DATA_FILES[split] is None]
assert not missing, f"Download did not produce these files: {missing}"

for split, path in DATA_FILES.items():
    print(f"{split:12s} {path} ({path.stat().st_size / 1e6:.1f} MB)")

## Load and tokenize PrimeVul

Each JSONL row must contain:

- `func`: the C/C++ function text
- `target`: `1` for vulnerable and `0` for benign
- `idx`: the sample identifier

The tokenizer truncates long functions to 512 tokens exactly as the official baseline does. Tokenization is cached on the Colab VM for the current session.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw = load_dataset(
    "json",
    data_files={split: str(path) for split, path in DATA_FILES.items()},
    cache_dir=str(LOCAL_HF_CACHE / "datasets"),
)

required_columns = {"func", "target", "idx"}
for split, dataset in raw.items():
    missing_columns = required_columns - set(dataset.column_names)
    assert not missing_columns, f"{split} is missing columns: {missing_columns}"
    labels = np.asarray(dataset["target"], dtype=np.int64)
    assert set(np.unique(labels)).issubset({0, 1}), f"Unexpected labels in {split}"
    print(
        f"{split:12s}: {len(dataset):7,d} rows | "
        f"vulnerable={int(labels.sum()):6,d} | benign={int((labels == 0).sum()):7,d}"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=str(LOCAL_HF_CACHE / "models"),
)


def tokenize_batch(batch):
    encoded = tokenizer(
        batch["func"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    # Shape [batch, 1] makes Hugging Face use BCEWithLogitsLoss for one logit.
    encoded["labels"] = [[float(label)] for label in batch["target"]]
    return encoded


tokenized = raw.map(
    tokenize_batch,
    batched=True,
    batch_size=512,
    remove_columns=raw["train"].column_names,
    desc="Tokenizing PrimeVul",
)

example = tokenized["train"][0]
print("Tokenized columns:", tokenized["train"].column_names)
print("input_ids length:", len(example["input_ids"]), "labels:", example["labels"])

## Build the binary classifier

Yes—the standard baseline is binary cross-entropy. UniXcoder supplies the encoder, and its classification head emits one logit. Setting `problem_type="multi_label_classification"` with one label makes Transformers use numerically stable `BCEWithLogitsLoss`. This is equivalent to the official implementation's sigmoid followed by BCE.

No vulnerable-class weight is used here; that is a separate PrimeVul experiment (Table VI), not the standard baseline.

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, DefaultDataCollator

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="multi_label_classification",
    cache_dir=str(LOCAL_HF_CACHE / "models"),
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    cache_dir=str(LOCAL_HF_CACHE / "models"),
)

data_collator = DefaultDataCollator()
model.to("cuda")

# Two-sample forward-pass check before committing to the long training run.
smoke_batch = data_collator([tokenized["train"][0], tokenized["train"][1]])
smoke_batch = {key: value.to("cuda") for key, value in smoke_batch.items()}
model.eval()
with torch.no_grad(), torch.autocast(
    device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16
):
    smoke_output = model(**smoke_batch)

print("Smoke-test loss:", float(smoke_output.loss))
print("Logit shape:", tuple(smoke_output.logits.shape))
assert smoke_output.logits.shape == (2, 1)
del smoke_batch, smoke_output
torch.cuda.empty_cache()

In [ ]:
from scipy.special import expit
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_curve


def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    scores = expit(np.asarray(logits).reshape(-1))
    labels = np.asarray(eval_prediction.label_ids).reshape(-1).astype(np.int64)
    predictions = (scores >= THRESHOLD).astype(np.int64)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    }


def calculate_vd_score(labels, scores, target_fpr=0.005):
    """Official PrimeVul definition: false-negative rate at FPR <= 0.5%."""
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    valid_indices = np.where(fpr <= target_fpr)[0]
    index = valid_indices[-1] if len(valid_indices) else np.abs(fpr - target_fpr).argmin()
    return float(1.0 - tpr[index]), float(thresholds[index]), float(fpr[index])


def calculate_pair_metrics(targets, predictions):
    """Treat each adjacent two rows in PrimeVul's paired file as one pair."""
    targets = np.asarray(targets, dtype=np.int64)
    predictions = np.asarray(predictions, dtype=np.int64)
    assert len(targets) == len(predictions) and len(targets) % 2 == 0

    pair_targets = targets.reshape(-1, 2)
    pair_predictions = predictions.reshape(-1, 2)
    if not np.all(pair_targets.sum(axis=1) == 1):
        bad = np.where(pair_targets.sum(axis=1) != 1)[0][:5]
        raise ValueError(f"Expected one vulnerable and one benign row per adjacent pair; bad pairs: {bad}")

    vulnerable_column = pair_targets.argmax(axis=1)
    benign_column = 1 - vulnerable_column
    rows = np.arange(len(pair_targets))
    vulnerable_pred = pair_predictions[rows, vulnerable_column]
    benign_pred = pair_predictions[rows, benign_column]

    outcomes = {
        "P-C": (vulnerable_pred == 1) & (benign_pred == 0),
        "P-V": (vulnerable_pred == 1) & (benign_pred == 1),
        "P-B": (vulnerable_pred == 0) & (benign_pred == 0),
        "P-R": (vulnerable_pred == 0) & (benign_pred == 1),
    }
    result = {name: float(mask.mean() * 100.0) for name, mask in outcomes.items()}
    result["pairs"] = int(len(pair_targets))
    assert abs(sum(result[name] for name in ("P-C", "P-V", "P-B", "P-R")) - 100.0) < 1e-6
    return result


PUBLISHED_UNIXCODER_PV_PV = {
    "accuracy": 96.86,
    "f1": 21.43,
    "VD-S": 89.21,
    "P-C": 1.60,
    "P-V": 12.06,
    "P-B": 85.11,
    "P-R": 1.24,
}

## Configure training and checkpoint recovery

Training all 237k functions for 10 epochs is a real full fine-tuning run and may take several hours. Validation and checkpointing occur every 1,000 optimizer steps. Only a few checkpoints are retained, including the best validation-F1 checkpoint.

If Colab disconnects, reconnect to an A100, rerun the notebook, and this section will detect the newest checkpoint in Google Drive. The next cell then resumes optimizer, scheduler, and model state automatically.

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=False,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.0,
    max_grad_norm=1.0,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    bf16=USE_BF16,
    tf32=False,
    optim="adamw_torch",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=EVAL_AND_SAVE_STEPS,
    save_steps=EVAL_AND_SAVE_STEPS,
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to=["tensorboard"],
    remove_unused_columns=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
resume_checkpoint = last_checkpoint if RESUME_FROM_LAST_CHECKPOINT else None
print("Resume checkpoint:", resume_checkpoint or "none; starting from Hugging Face weights")

run_config = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": WARMUP_STEPS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "eval_and_save_steps": EVAL_AND_SAVE_STEPS,
    "precision": "BF16" if USE_BF16 else "FP32",
    "loss": "BCEWithLogitsLoss (unweighted)",
}
with open(PROJECT_DIR / "run_config.json", "w") as file:
    json.dump(run_config, file, indent=2)

In [ ]:
# This is the long-running cell.
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

# Because load_best_model_at_end=True, trainer.model now holds the checkpoint with
# the highest validation F1, not merely the final epoch's weights.
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
trainer.save_state()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation F1:", trainer.state.best_metric)
print("Exported model:", FINAL_MODEL_DIR)

## Evaluate the best checkpoint

The ordinary test set provides accuracy, precision, recall, F1, and VD-S. The paired test set is evaluated as adjacent vulnerable/patched pairs:

- **P-C:** vulnerable detected and patch labeled benign
- **P-V:** both labeled vulnerable
- **P-B:** both labeled benign
- **P-R:** vulnerable labeled benign and patch labeled vulnerable

The four paired percentages must sum to 100%.

In [ ]:
# Full-test predictions.
test_output = trainer.predict(tokenized["test"], metric_key_prefix="test")
test_logits = np.asarray(test_output.predictions).reshape(-1)
test_scores = expit(test_logits)
test_targets = np.asarray(raw["test"]["target"], dtype=np.int64)
test_predictions = (test_scores >= THRESHOLD).astype(np.int64)

vd_score, vd_threshold, realized_fpr = calculate_vd_score(test_targets, test_scores)
full_test_metrics = {
    "accuracy": accuracy_score(test_targets, test_predictions) * 100.0,
    "precision": precision_score(test_targets, test_predictions, zero_division=0) * 100.0,
    "recall": recall_score(test_targets, test_predictions, zero_division=0) * 100.0,
    "f1": f1_score(test_targets, test_predictions, zero_division=0) * 100.0,
    "VD-S": vd_score * 100.0,
    "VD-S threshold": vd_threshold,
    "VD-S realized FPR": realized_fpr * 100.0,
}

# Paired-test predictions. Pair metrics use the ordinary 0.5 threshold.
paired_output = trainer.predict(tokenized["test_paired"], metric_key_prefix="paired")
paired_logits = np.asarray(paired_output.predictions).reshape(-1)
paired_scores = expit(paired_logits)
paired_targets = np.asarray(raw["test_paired"]["target"], dtype=np.int64)
paired_predictions = (paired_scores >= THRESHOLD).astype(np.int64)
pair_metrics = calculate_pair_metrics(paired_targets, paired_predictions)

reproduced = {**full_test_metrics, **pair_metrics}
comparison_rows = ["accuracy", "f1", "VD-S", "P-C", "P-V", "P-B", "P-R"]
comparison = pd.DataFrame(
    {
        "published (%)": [PUBLISHED_UNIXCODER_PV_PV[name] for name in comparison_rows],
        "this run (%)": [reproduced[name] for name in comparison_rows],
    },
    index=comparison_rows,
)

print("Full-test metrics:")
print(json.dumps(full_test_metrics, indent=2))
print("\nPaired metrics:")
print(json.dumps(pair_metrics, indent=2))
print("\nPublished baseline versus this run:")
display(comparison.round(2))

In [ ]:
# Save reproducible outputs to Google Drive.
full_predictions_df = pd.DataFrame(
    {
        "idx": raw["test"]["idx"],
        "target": test_targets,
        "prediction": test_predictions,
        "vulnerability_score": test_scores,
    }
)
paired_predictions_df = pd.DataFrame(
    {
        "idx": raw["test_paired"]["idx"],
        "target": paired_targets,
        "prediction": paired_predictions,
        "vulnerability_score": paired_scores,
        "pair_number": np.repeat(np.arange(len(paired_targets) // 2), 2),
    }
)

full_predictions_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
paired_predictions_df.to_csv(RESULTS_DIR / "paired_test_predictions.csv", index=False)

# Official PrimeVul-compatible prediction format: idx, binary prediction, score.
full_predictions_df[["idx", "prediction", "vulnerability_score"]].to_csv(
    RESULTS_DIR / "predictions.txt", sep="\t", index=False, header=False
)

summary = {
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_f1": trainer.state.best_metric,
    "full_test_metrics": full_test_metrics,
    "pair_metrics": pair_metrics,
    "published_unixcoder_pv_pv": PUBLISHED_UNIXCODER_PV_PV,
    "run_config": run_config,
}
with open(RESULTS_DIR / "metrics.json", "w") as file:
    json.dump(summary, file, indent=2)

print("Saved model and tokenizer to:", FINAL_MODEL_DIR)
print("Saved metrics and predictions to:", RESULTS_DIR)
print("Checkpoint recovery directory:", CHECKPOINT_DIR)

## What is saved

Everything important persists in `MyDrive/primevul_unixcoder_baseline`:

- `data/`: original PrimeVul JSONL files
- `checkpoints/`: resumable Trainer checkpoints (model, optimizer, scheduler, RNG, and Trainer state)
- `best_model/`: exported best-validation-F1 model and tokenizer
- `results/metrics.json`: full and paired metrics
- `results/test_predictions.csv`: full-test labels, predictions, and probabilities
- `results/paired_test_predictions.csv`: paired-test outputs
- `results/predictions.txt`: format accepted by PrimeVul's official VD-S script

If CUDA reports out-of-memory, lower `TRAIN_BATCH_SIZE` from 16 to 8 and raise `GRADIENT_ACCUMULATION_STEPS` from 4 to 8 so the effective batch remains 64. If a partial dataset download fails, delete only the incomplete files under the notebook's Drive `data/` directory and rerun the download cell.